In [1]:
import os
import json
from tqdm import tqdm
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import HDBSCAN, OPTICS
from sklearn.preprocessing import normalize
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
from hdbscan.validity import validity_index
import warnings
warnings.filterwarnings("ignore")

from extraction_prompts import (
    FIND_BETTER_ENTITY_NAME,
    DECIDE_OUTLIER_FATE_ENTITY,
    DECIDE_OUTLIER_FATE_RELATION,
    FIND_BETTER_RELATION_NAME,
)

from utils import ollama_request, EMBEDD_MODEL_1

INDEX_NAME = os.getenv("INDEX_NAME")

/home/zbrzeznyg/miniconda3/envs/masters/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def calc_dbcv(X, labels):
    X = np.array(X)
    distance_matrix = pairwise_distances(X, metric="cosine")

    return validity_index(
        distance_matrix,
        labels,
        metric="precomputed",
        d=X.shape[1]
    )

def calc_semantic_cohesion(embeddings):
    n_obs = embeddings.shape[0]
    cosine_matrix = embeddings @ embeddings.T
    cos_sum = cosine_matrix[np.triu_indices(len(embeddings), k=1)].sum()

    return 2 * cos_sum / (n_obs * (n_obs - 1))


In [7]:
def validate_terms(mapping_path, data, outlier_prompt):
    # if os.path.exists(mapping_path):
    #     with open(mapping_path, "r") as f:
    #         mapping = json.load(f)
    # else:
    #     mapping = {}
    mapping = {}
    for term in data:
        # if str(term) in mapping:
        #     continue
        # is_valid_judge_1 = ollama_request(
        #     prompt=outlier_prompt.format(outlier=term),
        #     is_stream=False,
        #     model_name="ministral-3:3b"
        # ).replace('"', "").replace("'", "").replace("*", "").strip().lower()
        # is_valid_judge_2 = ollama_request(
        #     prompt=outlier_prompt.format(outlier=term),
        #     is_stream=False,
        #     model_name="llama3.2:latest"
        # ).replace('"', "").replace("'", "").replace("*", "").strip().lower()
        # if is_valid_judge_1 == "invalid" and is_valid_judge_2 == "invalid":
        #     mapping[str(term)] = None
        # else:
        #     mapping[str(term)] = str(term)
        mapping[str(term)] = str(term)
    with open(mapping_path, "w") as f:
        json.dump(mapping, f, indent=4)


def create_data_mapping(mapping_path, embed_model, cluster_model, replace_prompt, model_name, N=2):
    with open(mapping_path, "r") as f:
        mapping = json.load(f)
    valid_data = [k for k, v in mapping.items() if v is not None]
    encoded_entities = embed_model.encode(valid_data, normalize_embeddings=True).tolist()
    try:
        cluster_labels = cluster_model.fit_predict(encoded_entities)
    except Exception as e:
        outliers = [-1 for _ in range(len(valid_data))]
    outliers = np.array(valid_data)[np.where(cluster_labels == -1)[0]]
    for outlier in outliers:
        mapping[str(outlier)] = str(outlier)

    distances_to_new = []
    semantic_cohesion_scores = []

    for i in tqdm(range(max(cluster_labels) + 1), desc="Finding better names..."):
        cluster_indices = np.where(cluster_labels == i)[0]
        cluster_entities = np.array(valid_data)[cluster_indices]
        encoded_cluster_entities = np.array(encoded_entities)[cluster_indices]
        cluster_mean = np.mean(encoded_cluster_entities, axis=0)
        distances = np.linalg.norm(encoded_cluster_entities - cluster_mean, axis=1)
        closest_entities = cluster_entities[distances <= min(distances)+(max(distances) - min(distances))/N]
        better_entity_name = ollama_request(
            prompt=replace_prompt.format(list=closest_entities.tolist()),
            is_stream=False,
            model_name=model_name
        ).strip().lower().replace('"', "").replace("'", "").replace("*", "")

        if "do_not_merge" in better_entity_name:
            # print(closest_entities.tolist())
            for entity in cluster_entities:
                mapping[str(entity)] = str(entity)
            continue

        # encoded_better_entity = embed_model.encode(better_entity_name, normalize_embeddings=True)
        # distances_to_better_entity = np.mean(np.linalg.norm(encoded_cluster_entities - encoded_better_entity, axis=1))
        # distances_to_new.append(distances_to_better_entity)
        # semantic_cohesion_scores.append(calc_semantic_cohesion(encoded_cluster_entities))

        for entity in cluster_entities:
            mapping[str(entity)] = better_entity_name

    # metric_scores = {
    #     "dbcv_score": round(float(calc_dbcv(encoded_entities, cluster_labels)), 4),
    #     "distances_to_new": round(float(np.mean(np.array(distances_to_new))), 4),
    #     "cohesion_score": round(float(np.mean(np.array(semantic_cohesion_scores))), 4),
    #     "outliers_ratio": round(len(outliers) / len(valid_data), 4)
    # }

    with open(mapping_path, "w") as f:
        json.dump(mapping, f, indent=4)
    return mapping, valid_data, encoded_entities, cluster_labels#, metric_scores

In [8]:
def viz_clusters(valid_data, encoded_entities, cluster_labels):
    embeddings = np.array(encoded_entities)
    cluster_labels = np.array(cluster_labels)

    embeddings_2d = TSNE(
        n_components=2,
        perplexity=30,
        random_state=42
    ).fit_transform(embeddings)

    plot_df = pd.DataFrame({
        "x": embeddings_2d[:, 0],
        "y": embeddings_2d[:, 1],
        "name": valid_data,
        "cluster": cluster_labels
    })

    plot_df["cluster"] = plot_df["cluster"].astype(str)

    fig = px.scatter(
        plot_df,
        x="x",
        y="y",
        color="cluster",
        color_discrete_map={
            "-1": "black"
        },
        hover_name="name",
        hover_data={
            "cluster": True,
            "x": False,
            "y": False
        },
    )

    fig.update_traces(
        marker=dict(size=8)
    )

    for trace in fig.data:
        if trace.name == "-1":
            trace.showlegend = True
            trace.name = "Outlier"
        else:
            trace.showlegend = False

    fig.update_layout(
        xaxis_title="t-SNE 1",
        yaxis_title="t-SNE 2",
        legend_title=""
    )

    fig.show()

### Final Results

In [9]:
triplets = pd.read_csv("data/triplets_extracted_100.csv")

#### Entities

In [ ]:
unique_entities = list(set(pd.concat([triplets["subject"], triplets["object"]]).dropna()))
embed_model_entities = EMBEDD_MODEL_1

cluster_model_entities = HDBSCAN(
    min_cluster_size=2,
    min_samples=2,
    metric="cosine",
    cluster_selection_method="eom",
)

outlier_prompt_entity = DECIDE_OUTLIER_FATE_ENTITY
replace_prompt_entity = FIND_BETTER_ENTITY_NAME
N_entities = 2

validate_terms(
    mapping_path="data/entities_mapping.json",
    data=unique_entities,
    outlier_prompt=outlier_prompt_entity
)

entities_mapping, valid_data_entities, encoded_entities, cluster_labels_entities = create_data_mapping(
    mapping_path="data/entities_mapping.json",
    embed_model=embed_model_entities,
    cluster_model=cluster_model_entities,
    replace_prompt=replace_prompt_entity,
    model_name="qwen2.5:3b",
    N=N_entities,
)

In [10]:
viz_clusters(valid_data_entities, encoded_entities, cluster_labels_entities)

#### Predicates

In [11]:
unique_predicates = list(set(triplets["predicate"].dropna()))
embed_model_predicates = EMBEDD_MODEL_1

cluster_model_predicates = OPTICS(
    min_samples=2,
    min_cluster_size=2,
    metric="cosine",
    cluster_method="xi",
    xi=0.02,
)

outlier_prompt_predicate = DECIDE_OUTLIER_FATE_RELATION
replace_prompt_predicate = FIND_BETTER_RELATION_NAME
N_predicates = 2

validate_terms(
    mapping_path="data/predicate_mapping.json",
    data=unique_predicates,
    outlier_prompt=outlier_prompt_predicate
)


predicate_mapping, valid_data_predicates, encoded_predicates, cluster_labels_predicates  = create_data_mapping(
    mapping_path="data/predicate_mapping.json",
    embed_model=embed_model_predicates,
    cluster_model=cluster_model_predicates,
    replace_prompt=replace_prompt_predicate,
    model_name="llama3.2:latest",
    N=N_predicates,
)

Finding better names...: 100%|██████████| 285/285 [01:17<00:00,  3.69it/s]


In [12]:
viz_clusters(valid_data_predicates, encoded_predicates, cluster_labels_predicates)

#### Trimm triplets

In [13]:
entities_mapping = json.load(open("data/entities_mapping.json", "r"))
predicate_mapping = json.load(open("data/predicate_mapping.json", "r"))

new_triplets = pd.DataFrame(columns=triplets.columns)
for t in tqdm(range(triplets.shape[0]), "Trimming entities..."):
    triplet = triplets.iloc[t].copy()
    new_subject = entities_mapping.get(str(triplet["subject"]), None)
    new_object = entities_mapping.get(str(triplet["object"]), None)
    new_predicate = predicate_mapping.get(str(triplet["predicate"]), None)
    if not new_subject or not new_object or not new_predicate:
        continue
    else:
        triplet["subject"] = new_subject
        triplet["object"] = new_object
        triplet["predicate"] = new_predicate

    new_triplets = pd.concat([new_triplets, pd.DataFrame([triplet])], ignore_index=True)

new_triplets = new_triplets.drop_duplicates()
new_triplets.to_csv("data/new_triplets_100.csv", index=False)

Trimming entities...: 100%|██████████| 1692/1692 [00:00<00:00, 2889.98it/s]


### Tests

In [33]:
sample_triplets = pd.read_csv("data/triplets_ministral_custom_chunk.csv")
mapping_path = "data/sample_entities_mapping.json"
unique_sample_entities = list(set(sample_triplets["predicate"].dropna()))
embed_model_entities = EMBEDD_MODEL_1

MODELS_TO_TEST = {
    "qwen": "qwen2.5:3b",
    "ministral": "ministral-3:3b",
    "llama": "llama3.2:latest",
}

OPTICS_CONFIGS = {
    "exp_1": {
        "min_samples": 2,
        "min_cluster_size": 2,
        "xi": 0.02,
    },
    "exp_2": {
        "min_samples": 2,
        "min_cluster_size": 2,
        "xi": 0.05,
    },
    "exp_3": {
        "min_samples": 2,
        "min_cluster_size": 3,
        "xi": 0.02,
    },
    "exp_4": {
        "min_samples": 3,
        "min_cluster_size": 2,
        "xi": 0.02,
    },
}

HDBSCAN_CONFIGS = {
    "exp_1": {
        "min_samples": 2,
        "min_cluster_size": 2,
        "cluster_selection_method": "leaf",
    },
    "exp_2": {
        "min_samples": 2,
        "min_cluster_size": 3,
        "cluster_selection_method": "leaf",
    },
    "exp_3": {
        "min_samples": 3,
        "min_cluster_size": 2,
        "cluster_selection_method": "leaf",
    },
    "exp_4": {
        "min_samples": 2,
        "min_cluster_size": 2,
        "cluster_selection_method": "eom",
    },
}

results = {}
results["optics"] = {}
results["hdbscan"] = {}

for model in MODELS_TO_TEST:
    results["optics"][model] = {}
    results["hdbscan"][model] = {}
    for exp in OPTICS_CONFIGS:
        print(f"Running {model} with {exp}...")
        cluster_model = OPTICS(
            min_samples=OPTICS_CONFIGS[exp]["min_samples"],
            min_cluster_size=OPTICS_CONFIGS[exp]["min_cluster_size"],
            metric="cosine",
            cluster_method="xi",
            xi=OPTICS_CONFIGS[exp]["xi"],
        )

        validate_terms(
            mapping_path=mapping_path,
            data=unique_sample_entities,
            outlier_prompt=DECIDE_OUTLIER_FATE_ENTITY
        )

        entities_mapping, valid_data_entities, encoded_entities, cluster_labels_entities, metric_scores = create_data_mapping(
            mapping_path=mapping_path,
            embed_model=embed_model_entities,
            cluster_model=cluster_model,
            replace_prompt=FIND_BETTER_ENTITY_NAME,
            model_name=MODELS_TO_TEST[model],
            N=2,
        )

        results["optics"][model][exp] = metric_scores

    for exp in HDBSCAN_CONFIGS:
        print(f"Running {model} with {exp}...")
        cluster_model = HDBSCAN(
            min_cluster_size=HDBSCAN_CONFIGS[exp]["min_cluster_size"],
            min_samples=HDBSCAN_CONFIGS[exp]["min_samples"],
            metric="cosine",
            cluster_selection_method=HDBSCAN_CONFIGS[exp]["cluster_selection_method"],
        )

        validate_terms(
            mapping_path=mapping_path,
            data=unique_sample_entities,
            outlier_prompt=DECIDE_OUTLIER_FATE_ENTITY
        )

        entities_mapping, valid_data_entities, encoded_entities, cluster_labels_entities, metric_scores = create_data_mapping(
            mapping_path=mapping_path,
            embed_model=embed_model_entities,
            cluster_model=cluster_model,
            replace_prompt=FIND_BETTER_ENTITY_NAME,
            model_name=MODELS_TO_TEST[model],
            N=2,
        )

        results["hdbscan"][model][exp] = metric_scores

Running qwen with exp_1...


Finding better names...: 100%|██████████| 293/293 [01:10<00:00,  4.15it/s]


Running qwen with exp_2...


Finding better names...: 100%|██████████| 263/263 [01:00<00:00,  4.34it/s]


Running qwen with exp_3...


...

Finding better names...: 100%|██████████| 152/152 [00:40<00:00,  3.77it/s]


Running llama with exp_3...


Finding better names...: 100%|██████████| 151/151 [00:40<00:00,  3.74it/s]


Running llama with exp_4...


Finding better names...: 100%|██████████| 306/306 [01:20<00:00,  3.80it/s]


In [35]:
independent_metrics = pd.DataFrame.from_dict(results['hdbscan']["qwen"], orient="index").reset_index()
independent_metrics.rename(columns={'index': 'experiment'}, inplace=True)
independent_metrics[["experiment", "dbcv_score", "outliers_ratio"]]

,experiment,dbcv_score,outliers_ratio
0,exp_1,0.2091,0.3634
1,exp_2,0.1370,0.4905
2,exp_3,0.1336,0.5980
3,exp_4,0.2134,0.3535


In [39]:
independent_metrics = pd.DataFrame.from_dict(results['hdbscan']["qwen"], orient="index").reset_index()
independent_metrics.rename(columns={'index': 'experiment'}, inplace=True)
independent_metrics[["experiment", "distances_to_new", "cohesion_score"]]

,experiment,distances_to_new,cohesion_score
0,exp_1,0.6248,0.7083
1,exp_2,0.7354,0.6473
2,exp_3,0.7056,0.6405
3,exp_4,0.6451,0.6797


In [40]:
independent_metrics = pd.DataFrame.from_dict(results['hdbscan']["llama"], orient="index").reset_index()
independent_metrics.rename(columns={'index': 'experiment'}, inplace=True)
independent_metrics[["experiment", "distances_to_new", "cohesion_score"]]

,experiment,distances_to_new,cohesion_score
0,exp_1,0.6358,0.6706
1,exp_2,0.7207,0.6091
2,exp_3,0.6694,0.6400
3,exp_4,0.6474,0.6676


In [41]:
independent_metrics = pd.DataFrame.from_dict(results['hdbscan']["ministral"], orient="index").reset_index()
independent_metrics.rename(columns={'index': 'experiment'}, inplace=True)
independent_metrics[["experiment", "distances_to_new", "cohesion_score"]]

,experiment,distances_to_new,cohesion_score
0,exp_1,0.6457,0.6836
1,exp_2,0.7209,0.6108
2,exp_3,0.6484,0.6503
3,exp_4,0.6634,0.6751


In [25]:
# dbcv score and outliers ratio does not rely on model thus they can be shown per experiment
# distances to new (because of the new value for each cluster) and cohesion score (as do not merge for different models can be in different clusters) are model dependent thus they can be shown per model and experiment

pd.DataFrame.from_dict(results['optics']["qwen"], orient="index").reset_index()

,index,dbcv_score,distances_to_new,cohesion_score,outliers_ratio
0,exp_1,0.2168,0.3941,0.7781,0.4176
1,exp_2,0.2159,0.3895,0.7857,0.4610
2,exp_3,0.1430,0.4853,0.7412,0.6311
3,exp_4,0.1376,0.5117,0.7248,0.6211
